# DA-AG-014 SQL and Advanced Functions


## Q1. Explain the fundamental differences between DDL, DML, and DQL commands in SQL.

DDL defines and changes database structure, DML manipulates data, and DQL queries data. Examples are CREATE for DDL, INSERT or UPDATE for DML, and SELECT for DQL.

## Q2. What is the purpose of SQL constraints?

Constraints enforce rules on data to improve accuracy and integrity. Common examples include PRIMARY KEY, FOREIGN KEY, UNIQUE, NOT NULL, and CHECK.

## Q3. Explain the difference between LIMIT and OFFSET clauses in SQL.

LIMIT controls how many rows are returned, while OFFSET skips a number of rows before returning results. Together they support pagination.

## Q4. What is a Common Table Expression (CTE) in SQL, and what are its main benefits?

A CTE is a named temporary result set defined with WITH. It improves readability, simplifies complex queries, and can be reused within the same query.

## Q5. Describe SQL normalization and its primary goals. Briefly explain 1NF, 2NF, 3NF.

Normalization organizes data to reduce redundancy and improve integrity. 1NF removes repeating groups, 2NF removes partial dependency, and 3NF removes transitive dependency.

## Q6. Create database ECommerceDB and define tables with constraints, then insert the given records.

Use CREATE TABLE statements with primary and foreign keys, then INSERT the provided sample records into Categories, Products, Customers, and Orders.


CREATE DATABASE ECommerceDB;
USE ECommerceDB;



CREATE TABLE Categories (
    CategoryID   INT PRIMARY KEY,
    CategoryName VARCHAR(50) NOT NULL UNIQUE
);

CREATE TABLE Products (
    ProductID     INT PRIMARY KEY,
    ProductName   VARCHAR(100) NOT NULL UNIQUE,
    CategoryID    INT,
    Price         DECIMAL(10,2) NOT NULL,
    StockQuantity INT,
    FOREIGN KEY (CategoryID) REFERENCES Categories(CategoryID)
);

CREATE TABLE Customers (
    CustomerID   INT PRIMARY KEY,
    CustomerName VARCHAR(100) NOT NULL,
    Email        VARCHAR(100) UNIQUE,
    JoinDate     DATE
);

CREATE TABLE Orders (
    OrderID     INT PRIMARY KEY,
    CustomerID  INT,
    OrderDate   DATE NOT NULL,
    TotalAmount DECIMAL(10,2),
    FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID)
);



INSERT INTO Categories VALUES
(1, 'Electronics'), (2, 'Books'), (3, 'Home Goods'), (4, 'Apparel');

INSERT INTO Products VALUES
(101, 'Laptop Pro',            1, 1200.00, 50),
(102, 'SQL Handbook',          2,   45.50, 200),
(103, 'Smart Speaker',         1,   99.99, 150),
(104, 'Coffee Maker',          3,   75.00, 80),
(105, 'Novel: The Great SQL',  2,   25.00, 120),
(106, 'Wireless Earbuds',      1,  150.00, 100),
(107, 'Blender X',             3,  120.00, 60),
(108, 'T-Shirt Casual',        4,   20.00, 300);

INSERT INTO Customers VALUES
(1, 'Alice Wonderland', 'alice@example.com',   '2023-01-10'),
(2, 'Bob the Builder',  'bob@example.com',     '2022-11-25'),
(3, 'Charlie Chaplin',  'charlie@example.com', '2023-03-01'),
(4, 'Diana Prince',     'diana@example.com',   '2021-04-26');

INSERT INTO Orders VALUES
(1001, 1, '2023-04-26', 1245.50),
(1002, 2, '2023-10-12',   99.99),
(1003, 1, '2023-07-01',  145.00),
(1004, 3, '2023-01-14',  150.00),
(1005, 2, '2023-09-24',  120.00),
(1006, 1, '2023-06-19',   20.00);

## Q7. Write SQL queries to retrieve all orders with customer and product details, and calculate the total bill for each order.

Use JOIN across Customers, Orders, and Products tables. Calculate the total bill as Quantity multiplied by Price for each order.

SELECT
    c.CustomerName,
    c.Email,
    COUNT(o.OrderID) AS TotalNumberofOrders
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CustomerName, c.Email
ORDER BY c.CustomerName;

## Q8. Retrieve ProductName, Price, StockQuantity, and CategoryName for all products ordered by category and product name.

Join Products with Categories, then sort by CategoryName and ProductName alphabetically.

SELECT
    p.ProductName,
    p.Price,
    p.StockQuantity,
    c.CategoryName
FROM Products p
JOIN Categories c ON p.CategoryID = c.CategoryID
ORDER BY c.CategoryName ASC, p.ProductName ASC;

## Q9. Use a CTE and window function to display the top 2 most expensive products in each category.

Use ROW_NUMBER() or RANK() partitioned by CategoryName and ordered by Price descending, then filter to the top two rows per category.

WITH RankedProducts AS (
    SELECT
        c.CategoryName,
        p.ProductName,
        p.Price,
        RANK() OVER (
            PARTITION BY p.CategoryID
            ORDER BY p.Price DESC
        ) AS price_rank
    FROM Products p
    JOIN Categories c ON p.CategoryID = c.CategoryID
)
SELECT
    CategoryName,
    ProductName,
    Price
FROM RankedProducts
WHERE price_rank <= 2
ORDER BY CategoryName, Price DESC;

## Q10. Sakila Video Rentals business questions.

1) 

SELECT
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    c.email,
    SUM(p.amount) AS total_amount_spent
FROM customer c
JOIN payment p ON c.customer_id = p.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.email
ORDER BY total_amount_spent DESC
LIMIT 5;

2) 



SELECT
    cat.name AS category_name,
    COUNT(r.rental_id) AS rental_count
FROM category cat
JOIN film_category fc ON cat.category_id = fc.category_id
JOIN film f ON fc.film_id = f.film_id
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
GROUP BY cat.category_id, cat.name
ORDER BY rental_count DESC
LIMIT 3;

3) 

SELECT
    i.store_id,
    COUNT(DISTINCT i.film_id) AS total_films_available,
    COUNT(DISTINCT CASE 
        WHEN i.inventory_id NOT IN (
            SELECT inventory_id FROM rental
        ) THEN i.film_id 
    END) AS films_never_rented
FROM inventory i
GROUP BY i.store_id;

4) 

SELECT
    YEAR(payment_date) AS year,
    MONTH(payment_date) AS month_number,
    MONTHNAME(payment_date) AS month_name,
    ROUND(SUM(amount), 2) AS total_revenue
FROM payment
WHERE YEAR(payment_date) = 2023
GROUP BY YEAR(payment_date), MONTH(payment_date), MONTHNAME(payment_date)
ORDER BY month_number;

5) 

SELECT
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    c.email,
    COUNT(r.rental_id) AS rental_count
FROM customer c
JOIN rental r ON c.customer_id = r.customer_id
WHERE r.rental_date >= DATE_SUB(CURDATE(), INTERVAL 6 MONTH)
GROUP BY c.customer_id, c.first_name, c.last_name, c.email
HAVING COUNT(r.rental_id) > 10
ORDER BY rental_count DESC;